In [1]:
import os
import gc
from typing import Tuple

import torch
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

2025-09-27 02:25:46.693140: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 02:25:46.880713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758939946.955822     751 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758939946.978389     751 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-27 02:25:47.181554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
IS_KAGGLE = os.path.exists("/kaggle/input")
TRAIN_DATA = "/kaggle/input/jigsaw-agile-community-rules/train.csv" if IS_KAGGLE else "data/train.csv"
TEST_DATA = "/kaggle/input/jigsaw-agile-community-rules/test.csv" if IS_KAGGLE else "data/test.csv"

## 前処理

In [3]:
## Data Cleaning
import nltk , emoji , re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer =  WordNetLemmatizer()

def clean_text(text: str) -> str:
    # 1. Convert emojis to text (keep them as tokens)
    text = emoji.demojize(text, delimiters=(" ", " "))

    # 2. Replace URLs with a placeholder
    text = re.sub(r'http\S+|www\.\S+', ' <URL> ', text)

    # 3. Replace mentions and hashtags (optional, if present in dataset)
    text = re.sub(r'@\w+', ' <USER> ', text)
    text = re.sub(r'#\w+', ' <HASHTAG> ', text)

    # 4. Normalize case
    text = text.lower()

    # 5. Keep only words and placeholders (remove other punctuation)
    text = re.sub(r'[^a-zA-Z0-9<> ]', ' ', text)

    # 6. Tokenize
#     tokens = word_tokenize(text)

    # 7. Lemmatize each token
#     lemmatized = [lemmatizer.lemmatize(token) for token in tokens if token.strip()]

    return text

## Rule + positive/negative_sample による対照学習

rule, positive/negative_sampleの関係性を学習する

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, DistilBertModel
import pandas as pd

class RuleContrastiveDataset(Dataset):
    """Rule Contrastive Learning用のデータセット"""

    def __init__(self, df, hp: dict) -> None:
        self.hp = hp

        self.tokenizer = AutoTokenizer.from_pretrained(self.hp['tokenizer'])
        self.max_length = self.hp['max_length']

        self.processed_texts = []
        for _, row in df.iterrows():
            # Rule text
            rule_text = row['rule']

            # Positive examples を結合
            combined_pos_text = f"{row['positive_example_1']} [SEP] {row['positive_example_2']}"

            # Negative examples を結合
            combined_neg_text = f"{row['negative_example_1']} [SEP] {row['negative_example_2']}"

            self.processed_texts.append({
                'rule_text': rule_text,
                'pos_text': combined_pos_text,
                'neg_text': combined_neg_text
            })

    def __len__(self) -> int:
        return len(self.processed_texts)

    def _tokenize(self, text_list: list[str]) -> list[torch.Tensor]:
        dataset = self.tokenizer(
            text_list,
            max_length=self.hp['max_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        gc.collect()

        return dataset

    def __getitem__(self, idx):
        sample = self.processed_texts[idx]
        rule_tokens = self._tokenize([sample['rule_text']])
        pos_tokens = self._tokenize([sample['pos_text']])
        neg_tokens = self._tokenize([sample['neg_text']])

        item =  {
            'rule_input_ids': rule_tokens['input_ids'].squeeze(0),
            'rule_attention_mask': rule_tokens['attention_mask'].squeeze(0),
            'pos_input_ids': pos_tokens['input_ids'].squeeze(0),
            'pos_attention_mask': pos_tokens['attention_mask'].squeeze(0),
            'neg_input_ids': neg_tokens['input_ids'].squeeze(0),
            'neg_attention_mask': neg_tokens['attention_mask'].squeeze(0),
        }


        return item

In [5]:
class RuleRepresentationModel(nn.Module):
    """Rule + positive/negative examples からrule表現を学習"""

    def __init__(self, bert_model_name, embed_dim=128):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(bert_model_name)
        self.projector = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def encode_text(self, input_ids, attention_mask):
        """テキストをエンコード"""
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = output.last_hidden_state[:, 0, :]  # [CLS]
        return self.projector(cls_embedding)

    def forward(self, batch):
        """バッチから直接embeddingを計算"""
        rule_emb = self.encode_text(batch['rule_input_ids'], batch['rule_attention_mask'])
        pos_emb = self.encode_text(batch['pos_input_ids'], batch['pos_attention_mask'])
        neg_emb = self.encode_text(batch['neg_input_ids'], batch['neg_attention_mask'])

        return rule_emb, pos_emb, neg_emb

def contrastive_loss_simple(rule_emb, pos_emb, neg_emb, temperature=0.1):
    """シンプルなcontrastive loss"""
    pos_sim = F.cosine_similarity(rule_emb, pos_emb, dim=1)
    pos_score = torch.exp(pos_sim / temperature)

    neg_sim = F.cosine_similarity(rule_emb, neg_emb, dim=1)
    neg_score = torch.exp(neg_sim / temperature)

    loss = -torch.log(pos_score / (pos_score + neg_score))
    return loss.mean()


In [6]:
# GPU使用率確認
def check_gpu_usage():
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name()}")
        print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    else:
        print("CUDA not available")

In [7]:
def train_rule_representation_simple(train_df, tokenizer, epochs=3, batch_size=16):
    """collate_fn不要の学習ループ"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    hp = {
        "base_model": "distilbert-base-uncased",
        "tokenizer": "distilbert-base-uncased",
        "max_length": 128,
        "lr": 1e-4,
        "batch_size": batch_size,
    }
    print(hp)
    dataset = RuleContrastiveDataset(train_df, hp=hp)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)  # collate_fn不要

    model = RuleRepresentationModel(hp["base_model"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"])

    model.train()

    for epoch in range(epochs):
        total_loss = 0
        num_batches = 0

        for batch_idx, batch in enumerate(dataloader):
            # バッチをGPUに移動
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass
            rule_emb, pos_emb, neg_emb = model(batch)

            # Loss calculation
            loss = contrastive_loss_simple(rule_emb, pos_emb, neg_emb)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if (batch_idx + 1) % 20 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx+1}, Loss: {loss.item():.4f}')

                check_gpu_usage()

        avg_loss = total_loss / max(num_batches, 1)
        print(f'Epoch {epoch+1}/{epochs} completed, Average Loss: {avg_loss:.4f}')

    return model

In [30]:
# tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# train_df = pd.read_csv(TRAIN_DATA)

# model_rpr = train_rule_representation_simple(train_df, tokenizer, epochs=3)

## table features

In [8]:
import re
import numpy as np
from sklearn.preprocessing import OneHotEncoder

def extract_tabular_features(df, encoder=None, is_train=True):
    # 文字数・単語数
    df['body_len'] = df['body'].apply(len)
    df['body_word_count'] = df['body'].apply(lambda x: len(str(x).split()))
    df['rule_len'] = df['rule'].apply(len)
    df['rule_word_count'] = df['rule'].apply(lambda x: len(str(x).split()))

    # 前処理によって情報がどれだけ保持されているか
    df['body_cleaned'] = df['body'].apply(clean_text)
    df['body_info_ratio'] = df['body_cleaned'].apply(lambda x: len(str(x).split())) / df['body_word_count'].replace(0, np.nan)

    # URL数
    df['body_url_count'] = df['body'].apply(lambda x: len(re.findall(r'http[s]?://', str(x))))

    # 絵文字数
    import emoji
    df['body_emoji_count'] = df['body'].apply(lambda x: len([c for c in str(x) if c in emoji.EMOJI_DATA]))

    # subreddit OneHot
    if is_train:
        ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')
        subreddit_ohe = ohe.fit_transform(df[['subreddit']])
        subreddit_ohe_df = pd.DataFrame(subreddit_ohe, columns=[f'subreddit_{cat}' for cat in ohe.categories_[0]])
        encoder_to_return = ohe
    else:
        if encoder is None:
            raise ValueError("Encoder must be provided for test/validation data")
        subreddit_ohe = encoder.transform(df[['subreddit']])
        subreddit_ohe_df = pd.DataFrame(subreddit_ohe, columns=[f'subreddit_{cat}' for cat in encoder.categories_[0]])
        encoder_to_return = encoder


    df = pd.concat([df.reset_index(drop=True), subreddit_ohe_df], axis=1)

    # 必要な特徴量のみ抽出
    tabular_features = [
        'body_len', 'body_word_count', 'rule_len', 'rule_word_count',
        'body_url_count', 'body_emoji_count', 'body_info_ratio'
    ] + list(subreddit_ohe_df.columns)

    return df[tabular_features], encoder_to_return

## table + bert model classifier

In [10]:
class HybridClassifierDataset(Dataset):
    """メモリ効率的なマルチモーダル分類用データセット"""

    def __init__(self, df, tabular_features, tokenizer, rule_model, max_length=256, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tabular_features = tabular_features
        self.tokenizer = tokenizer
        self.rule_model = rule_model
        self.max_length = max_length
        self.is_train = is_train

        # Rule modelをevalモードに設定
        self.rule_model.eval()

        # 事前処理：テキストクリーニングのみ
        self.processed_texts = []
        for _, row in df.iterrows():
            rule_text = clean_text(row['rule'])
            pos_text1 = clean_text(row['positive_example_1'])
            pos_text2 = clean_text(row['positive_example_2'])
            neg_text1 = clean_text(row['negative_example_1'])
            neg_text2 = clean_text(row['negative_example_2'])

            combined_pos_text = f"{pos_text1} [SEP] {pos_text2}"
            combined_neg_text = f"{neg_text1} [SEP] {neg_text2}"

            self.processed_texts.append({
                'rule_text': rule_text,
                'pos_text': combined_pos_text,
                'neg_text': combined_neg_text,
                'body_text': clean_text(row['body'])
            })

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_text = self.processed_texts[idx]

        # 1. Body tokenization
        body_tokens = self.tokenizer(
            sample_text['body_text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # 2. Rule representations（オンデマンド計算）
        with torch.no_grad():
            # Rule tokenization
            rule_tokens = self.tokenizer(
                sample_text['rule_text'],
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            pos_tokens = self.tokenizer(
                sample_text['pos_text'],
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            neg_tokens = self.tokenizer(
                sample_text['neg_text'],
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            # Rule model forward pass（GPU上で実行）
            device = next(self.rule_model.parameters()).device
            batch = {
                'rule_input_ids': rule_tokens['input_ids'].to(device),
                'rule_attention_mask': rule_tokens['attention_mask'].to(device),
                'pos_input_ids': pos_tokens['input_ids'].to(device),
                'pos_attention_mask': pos_tokens['attention_mask'].to(device),
                'neg_input_ids': neg_tokens['input_ids'].to(device),
                'neg_attention_mask': neg_tokens['attention_mask'].to(device)
            }

            rule_emb, pos_emb, neg_emb = self.rule_model(batch)

            # CPUに移動してdetach
            rule_emb = rule_emb.squeeze().detach().cpu()
            pos_emb = pos_emb.squeeze().detach().cpu()
            neg_emb = neg_emb.squeeze().detach().cpu()

        # 3. Tabular features
        tabular_feats = torch.FloatTensor(self.tabular_features.iloc[idx].values)

        item = {
            'body_input_ids': body_tokens['input_ids'].squeeze(0),
            'body_attention_mask': body_tokens['attention_mask'].squeeze(0),
            'tabular_feats': tabular_feats,
            'rule_emb': rule_emb,
            'pos_emb': pos_emb,
            'neg_emb': neg_emb
        }

        # Add labels for training
        if self.is_train and 'rule_violation' in row:
            item['labels'] = torch.LongTensor([row['rule_violation']]).squeeze()

        return item


In [11]:
class MultiModalClassifier(nn.Module):
    """5つの入力を統合したマルチモーダル分類器"""

    def __init__(self, bert_model_name, tabular_dim, rule_embed_dim=128, hidden_dim=256, dropout=0.2):
        super().__init__()

        # BERT for body encoding
        self.bert = DistilBertModel.from_pretrained(bert_model_name)

        # Body features projector
        self.body_projector = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Rule embeddings projectors
        self.rule_projector = nn.Sequential(
            nn.Linear(rule_embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.pos_projector = nn.Sequential(
            nn.Linear(rule_embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.neg_projector = nn.Sequential(
            nn.Linear(rule_embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Tabular features projector
        self.tabular_projector = nn.Sequential(
            nn.Linear(tabular_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Final classifier (5 inputs combined)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 5, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)  # 2-class classification
        )

    def forward(self, body_input_ids, body_attention_mask, tabular_feats, rule_emb, pos_emb, neg_emb):
        # 1. Body encoding
        body_output = self.bert(input_ids=body_input_ids, attention_mask=body_attention_mask)
        body_features = self.body_projector(body_output.last_hidden_state[:, 0, :])  # [CLS]

        # 2. Rule embeddings projection
        rule_features = self.rule_projector(rule_emb)
        pos_features = self.pos_projector(pos_emb)
        neg_features = self.neg_projector(neg_emb)

        # 3. Tabular features
        tabular_features = self.tabular_projector(tabular_feats)

        # 4. Combine all features
        combined_features = torch.cat([
            body_features,      # Body text
            rule_features,      # Rule representation
            pos_features,       # Positive examples
            neg_features,       # Negative examples
            tabular_features    # Tabular features
        ], dim=1)

        # 5. Final classification
        logits = self.classifier(combined_features)
        return logits

In [12]:
def predict_multimodal(model, test_df, tabular_features_test, tokenizer, rule_model):
    """マルチモーダル分類器での予測"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    test_dataset = HybridClassifierDataset(
        test_df, tabular_features_test, tokenizer, rule_model, is_train=False
    )
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            logits = model(
                batch['body_input_ids'],
                batch['body_attention_mask'],
                batch['tabular_feats'],
                batch['rule_emb'],
                batch['pos_emb'],
                batch['neg_emb']
            )

            probs = F.softmax(logits, dim=1)
            predictions.extend(probs[:, 1].cpu().numpy())

    return predictions

In [ ]:
# early stipping
import numpy as np
import torch

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pt'
            trace_func (function): trace print function.
                            Default: print
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [28]:
# 修正されたトレーニング関数
def train_multimodal_classifier_efficient(
        rule_model,
        train_df,
        val_df,
        tokenizer,
        tabular_features_train,
        tabular_features_val,
        epochs=5,
        batch_size=8
    ):
    """効率的なマルチモーダル分類器学習"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Rule modelをGPUに移動
    rule_model = rule_model.to(device)

    # 効率的なDatasets
    train_dataset = HybridClassifierDataset(
        train_df, tabular_features_train, tokenizer, rule_model, is_train=True
    )
    val_dataset = HybridClassifierDataset(
        val_df, tabular_features_val, tokenizer, rule_model, is_train=True
    )

    # DataLoaders with num_workers for efficiency
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)  # rule_modelがGPU上にあるためnum_workers=0
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    # Model
    tabular_dim = tabular_features_train.shape[1]
    model = MultiModalClassifier("distilbert-base-uncased", tabular_dim).to(device)

    # Training setup
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    early_stopping = EarlyStopping(patience=5, verbose=True)

    # Training loop
    best_auc = 0
    for epoch in range(epochs):
        # Training
        model.train()
        rule_model.eval()  # Rule modelは常にevalモード
        total_train_loss = 0
        total_val_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            # Move to device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            optimizer.zero_grad()

            # Forward pass
            logits = model(
                batch['body_input_ids'],
                batch['body_attention_mask'],
                batch['tabular_feats'],
                batch['rule_emb'],
                batch['pos_emb'],
                batch['neg_emb']
            )

            # Loss calculation
            loss = criterion(logits, batch['labels'])
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

            if (batch_idx + 1) % 20 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}')
                # check_gpu_usage()

        # Validation
        model.eval()
        val_predictions = []
        val_labels = []

        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

                logits = model(
                    batch['body_input_ids'],
                    batch['body_attention_mask'],
                    batch['tabular_feats'],
                    batch['rule_emb'],
                    batch['pos_emb'],
                    batch['neg_emb']
                )

                # early stopping用にval_lossも計算
                val_loss = criterion(logits, batch['labels'])
                total_val_loss += val_loss.item()

                probs = F.softmax(logits, dim=1)
                val_predictions.extend(probs[:, 1].cpu().numpy())
                val_labels.extend(batch['labels'].cpu().numpy())

        # Calculate metrics
        avg_loss = total_train_loss / len(train_loader)
        test_loss = total_val_loss / len(val_loader)
        val_auc = roc_auc_score(val_labels, val_predictions)

        print(f'Epoch {epoch+1}/{epochs}: Train Loss: {avg_loss:.4f}, Validation Loss: {test_loss:.4f}, Val AUC: {val_auc:.4f}')

        # # Save best model
        # if val_auc > best_auc:
        #     best_auc = val_auc
        #     torch.save(model.state_dict(), 'best_multimodal_model.pth')
        #     print(f'New best model saved with AUC: {best_auc:.4f}')

        scheduler.step()

        early_stopping(val_loss=test_loss, model=model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    return model

In [ ]:

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# データ読み込み
train_df = pd.read_csv(TRAIN_DATA)
test_df = pd.read_csv(TEST_DATA)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Training rule representation model...")
# Rule model training
rule_model = train_rule_representation_simple(train_df, tokenizer, epochs=3)

print("Extracting tabular features...")
# Tabular features抽出
train_tabular, tabular_encoder = extract_tabular_features(train_df.copy(), is_train=True)
test_tabular, _ = extract_tabular_features(test_df.copy(), encoder=tabular_encoder, is_train=False)

# 形状が一致することを確認
assert train_tabular.shape[1] == test_tabular.shape[1], f"Feature数が異なります: train={train_tabular.shape[1]}, test={test_tabular.shape[1]}"


Training rule representation model...
{'base_model': 'distilbert-base-uncased', 'tokenizer': 'distilbert-base-uncased', 'max_length': 128, 'lr': 0.0001, 'batch_size': 16}
Epoch 1/1, Batch 20, Loss: 0.1000
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.93 GB
Epoch 1/1, Batch 40, Loss: 0.0087
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.95 GB
Epoch 1/1, Batch 60, Loss: 0.0208
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.95 GB
Epoch 1/1, Batch 80, Loss: 0.0064
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.95 GB
Epoch 1/1, Batch 100, Loss: 0.0009
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.95 GB
Epoch 1/1, Batch 120, Loss: 0.0000
GPU: NVIDIA GeForce RTX 3060 Ti
Memory allocated: 1.02 GB
Memory reserved: 2.95 GB
Epoch 1/1 completed, Average Loss: 0.0496
Extracting tabular features...


In [29]:

# Train/Val split
train_idx, val_idx = train_test_split(
    range(len(train_df)), test_size=0.2, random_state=42,
    stratify=train_df['rule_violation']
)

train_df_split = train_df.iloc[train_idx].reset_index(drop=True)
val_df_split = train_df.iloc[val_idx].reset_index(drop=True)
train_tabular_split = train_tabular.iloc[train_idx].reset_index(drop=True)
val_tabular_split = train_tabular.iloc[val_idx].reset_index(drop=True)

print("Training multimodal classifier...")
# Multimodal classifier学習
multimodal_model = train_multimodal_classifier_efficient(
    rule_model, train_df_split, val_df_split, tokenizer,
    train_tabular_split, val_tabular_split, epochs=40, batch_size=16
)

print("Making predictions...")
# 予測
predictions = predict_multimodal(
    multimodal_model, test_df, test_tabular, tokenizer, rule_model
)
# 提出ファイル作成
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'rule_violation': predictions
})
submission_df.to_csv('submission.csv', index=False)
print("Submission file created: multimodal_submission.csv")

Training multimodal classifier...
Using device: cuda
Epoch 1/40, Batch 20/102, Loss: 0.6686
Epoch 1/40, Batch 40/102, Loss: 0.6173
Epoch 1/40, Batch 60/102, Loss: 0.7630
Epoch 1/40, Batch 80/102, Loss: 0.5590
Epoch 1/40, Batch 100/102, Loss: 0.6656
Epoch 1/40: Train Loss: 0.7029, Validation Loss: 0.6430, Val AUC: 0.7141
Validation loss decreased (inf --> 0.642987).  Saving model ...
Epoch 2/40, Batch 20/102, Loss: 0.6456
Epoch 2/40, Batch 40/102, Loss: 0.6347
Epoch 2/40, Batch 60/102, Loss: 0.4619
Epoch 2/40, Batch 80/102, Loss: 0.5369
Epoch 2/40, Batch 100/102, Loss: 0.6826
Epoch 2/40: Train Loss: 0.6166, Validation Loss: 0.5363, Val AUC: 0.8259
Validation loss decreased (0.642987 --> 0.536261).  Saving model ...
Epoch 3/40, Batch 20/102, Loss: 0.4539
Epoch 3/40, Batch 40/102, Loss: 0.5464
Epoch 3/40, Batch 60/102, Loss: 0.4783
Epoch 3/40, Batch 80/102, Loss: 0.3519
Epoch 3/40, Batch 100/102, Loss: 0.5550
Epoch 3/40: Train Loss: 0.4814, Validation Loss: 0.5117, Val AUC: 0.8517
Validat